In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r"C:\Users\shikh\OneDrive\Desktop\AI_Banking_system\Data\01_support_tickets.csv")

In [3]:
sentiment_df = df[['query_text','sentiment']]

In [4]:
sentiment_df2 = pd.read_csv(r"C:\Users\shikh\OneDrive\Desktop\AI_Banking_system\Data\synthetic_sentiment_queries.csv")

In [5]:
sentiment_df = pd.concat([sentiment_df, sentiment_df2], ignore_index=True)

In [6]:
import re
import pandas as pd

def clean_text(text):

    if pd.isna(text):
        return ""

    text = str(text).lower()

    # Contractions ko normalize karo
    contractions = {
        "didn't": "did not",
        "can't": "can not",
        "couldn't": "could not",
        "won't": "will not",
        "wouldn't": "would not",
        "isn't": "is not",
        "wasn't": "was not",
        "aren't": "are not",
        "don't": "do not",
        "doesn't": "does not",
        "haven't": "have not",
        "hasn't": "has not"
    }

    for contraction, replacement in contractions.items():
        text = text.replace(contraction, replacement)

    # URLs remove
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # Email remove
    text = re.sub(r'\S+@\S+', ' ', text)

    # Currency amounts remove
    # ₹999, ₹25,000, Rs 500, INR 1000 etc.
    text = re.sub(r'₹\s?[\d,]+', ' ', text)
    text = re.sub(r'\b(rs|inr)\.?\s?[\d,]+\b', ' ', text)

    # Remaining numbers remove
    text = re.sub(r'\d+', ' ', text)

    # Special characters remove
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # Extra spaces remove
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [7]:
sentiment_df["cleaned_query"] = sentiment_df["query_text"].apply(clean_text)

In [8]:
X = sentiment_df["cleaned_query"]
y = sentiment_df["sentiment"]

In [9]:
print(y.value_counts())

sentiment
Anxious       139
Confused      137
Urgent        133
Neutral       132
Frustrated    131
Angry         128
Name: count, dtype: int64


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_sentiment = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 3),
    sublinear_tf=True,
    min_df=1,
    max_df=0.95
)

X_train_tfidf = tfidf_sentiment.fit_transform(X_train)
X_test_tfidf = tfidf_sentiment.transform(X_test)

In [12]:
print("X_train:", X_train_tfidf.shape)
print("X_test:", X_test_tfidf.shape)

X_train: (640, 2108)
X_test: (160, 2108)


In [14]:
from sklearn.linear_model import LogisticRegression

In [15]:
sentiment_model = LogisticRegression(
    max_iter=2000,
    C=2.0,
    class_weight="balanced",
    random_state=42
)

sentiment_model.fit(X_train_tfidf, y_train)

y_pred = sentiment_model.predict(X_test_tfidf)

In [16]:
y_pred = sentiment_model.predict(X_test_tfidf)

print(y_pred[:10])

['Angry' 'Urgent' 'Frustrated' 'Anxious' 'Anxious' 'Confused' 'Frustrated'
 'Angry' 'Urgent' 'Frustrated']


In [17]:
print("Actual:")
print(y_test.head(10).values)

print("\nPredicted:")
print(y_pred[:10])

Actual:
['Angry' 'Urgent' 'Frustrated' 'Anxious' 'Anxious' 'Confused' 'Frustrated'
 'Angry' 'Urgent' 'Frustrated']

Predicted:
['Angry' 'Urgent' 'Frustrated' 'Anxious' 'Anxious' 'Confused' 'Frustrated'
 'Angry' 'Urgent' 'Frustrated']


In [18]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.8


In [19]:
results_df = pd.DataFrame({
    "query_text": X_test.values,
    "actual_sentiment": y_test.values,
    "predicted_sentiment": y_pred
})

print(results_df.head(20))

                                           query_text actual_sentiment  \
0   i am furious about this repeated problem pleas...            Angry   
1   please take immediate action on my request i n...           Urgent   
2   i can not complete video kyc the link keeps fa...       Frustrated   
3   i am nervous about this transaction please che...          Anxious   
4   i am worried about what will happen to my acco...          Anxious   
5   i do not understand the status of my request p...         Confused   
6   i am unhappy that my issue has not been fixed ...       Frustrated   
7   this service has made me extremely angry can t...            Angry   
8   please take immediate action on my request kin...           Urgent   
9   i am unhappy that my issue has not been fixed ...       Frustrated   
10  my company account kyc documents were submitte...          Anxious   
11  i do not understand what is happening with my ...         Confused   
12                   why was my neft t

In [20]:
wrong_predictions = results_df[
    results_df["actual_sentiment"] != results_df["predicted_sentiment"]
]

print(wrong_predictions)

                                            query_text actual_sentiment  \
10   my company account kyc documents were submitte...          Anxious   
18   i need to close my account but there are pendi...          Anxious   
26   there s an atm withdrawal of from bangalore bu...         Confused   
28          my card was charged by an unknown merchant         Confused   
31           i need to update my mobile number for kyc          Anxious   
38                 why is my account marked as dormant            Angry   
44                my emi was deducted twice this month          Anxious   
45    my fixed deposit matured but amount not credited           Urgent   
47   i updated my address but kyc is still showing ...         Confused   
49   multiple small transactions from my account i ...          Neutral   
52           my nri account kyc is expiring next month           Urgent   
55           my nri account kyc is expiring next month            Angry   
60                 my sal

In [21]:
import joblib
import os

os.makedirs("models", exist_ok=True)

# Save sentiment model
joblib.dump(sentiment_model, "models/sentiment_model.pkl")

# Save TF-IDF vectorizer
joblib.dump(tfidf_sentiment, "models/sentiment_tfidf.pkl")

print("Sentiment model saved successfully!")
print("TF-IDF vectorizer saved successfully!")

Sentiment model saved successfully!
TF-IDF vectorizer saved successfully!


Testing on nwe data points

In [22]:
import pandas as pd

new_queries = [
    "My account has been blocked and I don't know what to do",
    "Please stop this payment immediately",
    "I am really worried because my refund has not arrived",
    "I cannot understand why my verification keeps failing",
    "I am extremely upset with the service I received",
    "Could you tell me how I can update my bank details",
    "Someone accessed my account and I am scared about my money",
    "I have been waiting for days and this is getting annoying",
    "Please help me right now, this is very important",
    "I am not sure why my transaction is showing as pending",
    "I am furious that my issue has still not been resolved",
    "Can you explain what I need to do to complete this process",
    "I am worried that my payment may have gone to the wrong person",
    "This has been a terrible experience and I am very frustrated",
    "I need immediate assistance because my card is not working",
    "What documents do I need to submit for verification",
    "I don't understand why my account balance looks different",
    "I am concerned because I have not received my money yet",
    "This is unacceptable, I have contacted support multiple times",
    "Please provide information about the charges on my account"
]

# Clean the new queries using the SAME cleaning function
new_queries_cleaned = [clean_text(q) for q in new_queries]

# Transform using the SAME fitted TF-IDF
new_queries_tfidf = tfidf_sentiment.transform(new_queries_cleaned)

# Predict sentiment
predictions = sentiment_model.predict(new_queries_tfidf)

# Results
new_results = pd.DataFrame({
    "query_text": new_queries,
    "predicted_sentiment": predictions
})

print(new_results.to_string(index=False))

                                                    query_text predicted_sentiment
       My account has been blocked and I don't know what to do            Confused
                          Please stop this payment immediately              Urgent
         I am really worried because my refund has not arrived             Anxious
         I cannot understand why my verification keeps failing            Confused
              I am extremely upset with the service I received               Angry
            Could you tell me how I can update my bank details             Neutral
    Someone accessed my account and I am scared about my money             Anxious
     I have been waiting for days and this is getting annoying          Frustrated
              Please help me right now, this is very important              Urgent
        I am not sure why my transaction is showing as pending            Confused
        I am furious that my issue has still not been resolved          Frustrated
    